# XGBoost Experimentation — Vishing Detection

Pipeline dedicated to exploring XGBoost variations, the algorithm that showed the best performance in the multi-model pipeline of Notebook 7.

## Strategy
- **7 XGBoost variants** covering different hyperparameter axes
- Each variant is trained on **13 augmented-data datasets** (12 balanced + 1 raw 1M)
- Each variant is trained on **13 original-data datasets** (12 balanced + 1 raw 50K)
- Separate hold-out per data type (augmented: 200K | original: 10K)
- Models serialized as `VishingModelWrapper` in `s3://poc-fraude-vishing/proyecto/modelos_xgb/`

## XGBoost Variants
| Variant | Description |
|---|---|
| `xgb_base` | Same configuration as Notebook 7 (baseline) |
| `xgb_deep` | Deeper trees, more estimators, low lr |
| `xgb_shallow` | Shallow trees, many estimators (classic boosting) |
| `xgb_regularized` | Strong regularization (L1 + L2 + high min_child_weight) |
| `xgb_balanced` | `scale_pos_weight` adapted to the dataset's real imbalance |
| `xgb_conservative` | Aggressive subsampling (subsample + colsample_bytree + gamma) |
| `xgb_slow_learner` | Very low learning rate with 500 estimators |

In [ ]:
%pip install --quiet "sagemaker<3"

In [ ]:
import pandas as pd
import numpy as np
import copy
import json as _json
import joblib
import warnings
from io import BytesIO
from pathlib import Path
from urllib.parse import urlparse

import boto3
import sagemaker

from xgboost import XGBClassifier

from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, precision_recall_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings('ignore')
print('Libraries loaded successfully.')

## 1. Environment configuration

In [ ]:
sagemaker_session = sagemaker.Session()
role              = sagemaker.get_execution_role()
s3_client         = boto3.client('s3')

BUCKET       = 'poc-fraude-vishing'
BASE_PREFIX  = 'proyecto'
DATA_PREFIX  = f'{BASE_PREFIX}/data'
MODELS_PREFIX = f'{BASE_PREFIX}/modelos_xgb'   # new path, does not overwrite notebook 7

COLS_TO_DROP = [
    'session_id', 'customer_id', 'session_timestamp',
    'device_type', 'os_type', 'app_version',
    'biocatch_risk_score', 'biocatch_genuine_score',
    'biocatch_ato_indicator', 'biocatch_social_eng_indicator', 'biocatch_bot_indicator',
    'days_to_claim', 'claim_category',
    'screens_visited', 'unusual_screen_visits',
    'is_synthetic', 'interactions_per_s',
]

print(f'Bucket  : {BUCKET}')
print(f'Models  : s3://{BUCKET}/{MODELS_PREFIX}/')
print(f'Role    : {role}')

## 2. XGBoost variant definitions

Each dictionary entry defines the **fixed** hyperparameters of the variant.  
The `xgb_balanced` variant is special: its `scale_pos_weight` is computed at training time from the real imbalance of each dataset.

In [ ]:
COMMON = dict(
    tree_method='hist',
    device='cuda',
    eval_metric='logloss',
    random_state=42,
    use_label_encoder=False,
)

XGB_VARIANTS = {
    # Baseline identical to Notebook 7
    'xgb_base': dict(
        **COMMON,
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
    ),
    # Deeper trees + more iterations + low lr
    'xgb_deep': dict(
        **COMMON,
        n_estimators=300,
        max_depth=9,
        learning_rate=0.05,
        min_child_weight=3,
    ),
    # Classic boosting: simple trees, many iterations
    'xgb_shallow': dict(
        **COMMON,
        n_estimators=500,
        max_depth=3,
        learning_rate=0.05,
    ),
    # Strong regularization — reduces overfitting on synthetic data
    'xgb_regularized': dict(
        **COMMON,
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        reg_alpha=1.0,      # L1
        reg_lambda=5.0,     # L2
        min_child_weight=10,
        gamma=0.3,
    ),
    # scale_pos_weight computed dynamically per dataset
    # (marked with None here; replaced in the loop)
    'xgb_balanced': dict(
        **COMMON,
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        scale_pos_weight=None,  # placeholder — computed per dataset
    ),
    # Aggressive subsampling — stochastic regularization
    'xgb_conservative': dict(
        **COMMON,
        n_estimators=200,
        max_depth=5,
        learning_rate=0.08,
        subsample=0.7,
        colsample_bytree=0.7,
        gamma=0.5,
        min_child_weight=5,
    ),
    # Slow learning with many estimators
    'xgb_slow_learner': dict(
        **COMMON,
        n_estimators=500,
        max_depth=6,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
    ),
}

print(f'{len(XGB_VARIANTS)} variants defined:')
for name, params in XGB_VARIANTS.items():
    summary = {k: v for k, v in params.items()
               if k not in ('tree_method', 'device', 'eval_metric', 'random_state', 'use_label_encoder')}
    print(f'  {name:20s} → {summary}')

## 3. VishingModelWrapper
Reused from Notebook 7 without changes.

In [ ]:
class VishingModelWrapper:
    """
    Bundles a trained XGBoost model with its feature list and optimal
    decision threshold. Accepts dict, JSON string, or list[dict] as input.
    """

    def __init__(self, model, feature_names, threshold=0.5,
                 model_name='', technique='', ratio='', data_type=''):
        self.model         = model
        self.feature_names = list(feature_names)
        self.threshold     = threshold
        self.model_name    = model_name
        self.technique     = technique
        self.ratio         = ratio
        self.data_type     = data_type  # 'augmented' | 'original'

    def _to_array(self, json_input):
        if isinstance(json_input, str):
            data = _json.loads(json_input)
        elif isinstance(json_input, dict):
            data = json_input
        elif isinstance(json_input, list):
            return np.vstack([self._to_array(item) for item in json_input])
        else:
            raise TypeError(f'Expected dict, JSON string, or list. Got {type(json_input)}')
        missing = set(self.feature_names) - set(data.keys())
        if missing:
            raise ValueError(f'Missing features: {sorted(missing)}')
        return np.array([[data[f] for f in self.feature_names]], dtype=np.float64)

    def predict(self, json_input):
        proba  = self.model.predict_proba(self._to_array(json_input))[:, 1]
        labels = (proba >= self.threshold).astype(int).tolist()
        return labels[0] if len(labels) == 1 else labels

    def predict_proba_raw(self, json_input):
        proba = self.model.predict_proba(self._to_array(json_input))
        rows  = [{'legitimate': round(float(p[0]), 6), 'vishing': round(float(p[1]), 6)}
                 for p in proba]
        return rows[0] if len(rows) == 1 else rows

    def predict_full(self, json_input):
        proba   = self.model.predict_proba(self._to_array(json_input))
        results = []
        for p in proba:
            label = int(p[1] >= self.threshold)
            results.append({
                'prediction':             label,
                'label':                  'vishing' if label == 1 else 'legitimate',
                'probability_vishing':    round(float(p[1]), 6),
                'probability_legitimate': round(float(p[0]), 6),
                'threshold_used':         round(self.threshold, 6),
            })
        return results[0] if len(results) == 1 else results

    def __repr__(self):
        return (f'VishingModelWrapper(variant={self.model_name!r}, '
                f'data={self.data_type!r}, technique={self.technique!r}, '
                f'ratio={self.ratio!r}, n_features={len(self.feature_names)}, '
                f'threshold={self.threshold:.4f})')

## 4. Helper function: training a variant on a dataset

In [ ]:
def train_and_evaluate(variant_name, variant_params, X_train, y_train,
                       X_test, y_test, feature_names,
                       technique, ratio, data_type):
    """
    Trains an XGBoost variant, computes metrics on the holdout,
    serializes the wrapper, and uploads it to S3.
    Returns the metrics dict.
    """
    params = copy.deepcopy(variant_params)

    # xgb_balanced: scale_pos_weight computed from the training set
    if params.get('scale_pos_weight') is None:
        n_neg = int((y_train == 0).sum())
        n_pos = int((y_train == 1).sum())
        params['scale_pos_weight'] = round(n_neg / max(n_pos, 1), 2)

    model = XGBClassifier(**params)
    model.fit(X_train, y_train, verbose=False)

    y_prob = model.predict_proba(X_test)[:, 1]

    # Optimal threshold by max F1
    prec_arr, rec_arr, thr_arr = precision_recall_curve(y_test, y_prob)
    f1_arr  = 2 * prec_arr[:-1] * rec_arr[:-1] / (prec_arr[:-1] + rec_arr[:-1] + 1e-9)
    opt_thr = float(thr_arr[np.argmax(f1_arr)])
    y_pred  = (y_prob >= opt_thr).astype(int)

    # Package
    wrapper = VishingModelWrapper(
        model        = model,
        feature_names= feature_names,
        threshold    = opt_thr,
        model_name   = variant_name,
        technique    = technique,
        ratio        = ratio,
        data_type    = data_type,
    )

    # Upload to S3
    s3_key  = f'{MODELS_PREFIX}/{data_type}/{variant_name}/{technique}/{ratio}.pkl'
    buf     = BytesIO()
    joblib.dump(wrapper, buf)
    buf.seek(0)
    s3_client.upload_fileobj(buf, BUCKET, s3_key)

    return {
        'Variant'    : variant_name,
        'Data_Type'  : data_type,
        'Technique'  : technique,
        'Ratio'      : ratio,
        'Threshold'  : round(opt_thr, 4),
        'Recall'     : round(recall_score(y_test, y_pred), 4),
        'Precision'  : round(precision_score(y_test, y_pred), 4),
        'F1'         : round(f1_score(y_test, y_pred), 4),
        'ROC_AUC'    : round(roc_auc_score(y_test, y_prob), 4),
        'PR_AUC'     : round(average_precision_score(y_test, y_prob), 4),
        'S3_Path'    : f's3://{BUCKET}/{s3_key}',
        'scale_pos_weight': params.get('scale_pos_weight', 'N/A'),
    }

print('Function train_and_evaluate defined.')

## 5. Augmented Dataset (1M sessions)
### 5.1 Loading the augmented raw + hold-out extraction

In [ ]:
S3_AUG_PATH = f's3://{BUCKET}/{DATA_PREFIX}/augmented_data/dataset_1M_vishing_.parquet'
print(f'Loading: {S3_AUG_PATH}')

df_aug = pd.read_parquet(S3_AUG_PATH)
df_aug = df_aug.drop(columns=[c for c in COLS_TO_DROP if c in df_aug.columns])

X_aug = df_aug.drop(columns=['is_vishing'])
y_aug = df_aug['is_vishing']

_, X_test_aug, _, y_test_aug = train_test_split(
    X_aug, y_aug, test_size=0.20, random_state=42, stratify=y_aug
)

FEATURE_NAMES = X_test_aug.columns.tolist()

n = len(df_aug)
nv = int(y_aug.sum())
print(f'\nAugmented dataset  : {n:,} rows  |  vishing {nv/n*100:.2f}%')
print(f'Augmented hold-out : {len(X_test_aug):,} rows  |  vishing {y_test_aug.mean()*100:.2f}%')
print(f'Features           : {len(FEATURE_NAMES)}')

### 5.2 Listing of the 13 balanced augmented-data datasets

In [ ]:
prefix_balanced_aug = f'{DATA_PREFIX}/balanced/augmented/'
response = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=prefix_balanced_aug)

aug_paths = sorted([
    f"s3://{BUCKET}/{obj['Key']}"
    for obj in response.get('Contents', [])
    if obj['Key'].endswith('.parquet')
])

# Add augmented raw as dataset #13
aug_paths.append(S3_AUG_PATH)

print(f'Augmented-data datasets: {len(aug_paths)}')
for p in aug_paths:
    print(f'  {p}')

### 5.3 Training loop — augmented data
7 variants × 13 datasets = **91 models**

In [ ]:
results = []

total_aug = len(aug_paths) * len(XGB_VARIANTS)
done_aug  = 0

for path in aug_paths:
    parts     = urlparse(path).path.lstrip('/').split('/')
    technique = parts[-2]
    ratio     = Path(parts[-1]).stem

    df_train = pd.read_parquet(path)
    df_train = df_train.drop(columns=[c for c in COLS_TO_DROP if c in df_train.columns], errors='ignore')
    df_train = df_train.drop(index=X_test_aug.index, errors='ignore')  # exclude holdout rows

    X_tr = df_train.drop(columns=['is_vishing'])[FEATURE_NAMES]
    y_tr = df_train['is_vishing']

    for variant_name, variant_params in XGB_VARIANTS.items():
        done_aug += 1
        print(f'[{done_aug:3d}/{total_aug}] {technique:25s} | {ratio:30s} | {variant_name}', end=' ... ')

        row = train_and_evaluate(
            variant_name   = variant_name,
            variant_params = variant_params,
            X_train        = X_tr.values,
            y_train        = y_tr.values,
            X_test         = X_test_aug[FEATURE_NAMES].values,
            y_test         = y_test_aug.values,
            feature_names  = FEATURE_NAMES,
            technique      = technique,
            ratio          = ratio,
            data_type      = 'augmented',
        )
        results.append(row)
        print(f"Recall={row['Recall']:.3f}  PR-AUC={row['PR_AUC']:.3f}")

print('\n✅ Augmented-data training loop completed.')

## 6. Original Dataset (50K sessions)
### 6.1 Loading the original raw + hold-out extraction

In [ ]:
S3_ORIG_RAW_PATH = f's3://{BUCKET}/{BASE_PREFIX}/raw_data/biocatch_sinthetic_data.csv'
print(f'Loading: {S3_ORIG_RAW_PATH}')

df_orig = pd.read_csv(S3_ORIG_RAW_PATH)
df_orig = df_orig.drop(columns=[c for c in COLS_TO_DROP if c in df_orig.columns])

X_orig = df_orig.drop(columns=['is_vishing'])
y_orig = df_orig['is_vishing']

# Align to the same feature order as the augmented data
shared_feats = [f for f in FEATURE_NAMES if f in X_orig.columns]
X_orig = X_orig[shared_feats]

_, X_test_orig, _, y_test_orig = train_test_split(
    X_orig, y_orig, test_size=0.20, random_state=42, stratify=y_orig
)

FEATURE_NAMES_ORIG = shared_feats

n = len(df_orig)
nv = int(y_orig.sum())
print(f'\nOriginal dataset  : {n:,} rows  |  vishing {nv/n*100:.2f}%')
print(f'Original hold-out : {len(X_test_orig):,} rows  |  vishing {y_test_orig.mean()*100:.2f}%')
print(f'Features          : {len(FEATURE_NAMES_ORIG)}')

### 6.2 Listing of the 13 original-data datasets

In [ ]:
prefix_balanced_orig = f'{BASE_PREFIX}/data/balanced/original/'
response_orig = s3_client.list_objects_v2(Bucket=BUCKET, Prefix=prefix_balanced_orig)

orig_paths = sorted([
    f"s3://{BUCKET}/{obj['Key']}"
    for obj in response_orig.get('Contents', [])
    if obj['Key'].endswith('.csv')
])

# Add original raw as dataset #13
orig_paths.append(S3_ORIG_RAW_PATH)

print(f'Original-data datasets: {len(orig_paths)}')
for p in orig_paths:
    print(f'  {p}')

### 6.3 Training loop — original data
7 variants × 13 datasets = **91 models**

In [ ]:
total_orig = len(orig_paths) * len(XGB_VARIANTS)
done_orig  = 0

for path in orig_paths:
    parsed    = urlparse(path).path.lstrip('/')
    parts     = parsed.split('/')
    technique = parts[-2]
    ratio     = Path(parts[-1]).stem

    if path.endswith('.csv'):
        df_train = pd.read_csv(path)
    else:
        df_train = pd.read_parquet(path)

    df_train = df_train.drop(columns=[c for c in COLS_TO_DROP if c in df_train.columns], errors='ignore')

    # Exclude rows that belong to the original raw hold-out
    df_train = df_train.drop(index=X_test_orig.index, errors='ignore')

    X_tr = df_train.drop(columns=['is_vishing'])
    # Keep only shared features, in the same order
    X_tr = X_tr[[f for f in FEATURE_NAMES_ORIG if f in X_tr.columns]]
    y_tr = df_train['is_vishing']

    feat_names_ds = X_tr.columns.tolist()

    for variant_name, variant_params in XGB_VARIANTS.items():
        done_orig += 1
        print(f'[{done_orig:3d}/{total_orig}] {technique:25s} | {ratio:30s} | {variant_name}', end=' ... ')

        row = train_and_evaluate(
            variant_name   = variant_name,
            variant_params = variant_params,
            X_train        = X_tr.values,
            y_train        = y_tr.values,
            X_test         = X_test_orig[feat_names_ds].values,
            y_test         = y_test_orig.values,
            feature_names  = feat_names_ds,
            technique      = technique,
            ratio          = ratio,
            data_type      = 'original',
        )
        results.append(row)
        print(f"Recall={row['Recall']:.3f}  PR-AUC={row['PR_AUC']:.3f}")

print('\n✅ Original-data training loop completed.')

## 7. Results Analysis
### 7.1 Full table sorted by PR-AUC

In [ ]:
df_res = pd.DataFrame(results)

df_res_sorted = df_res.sort_values('PR_AUC', ascending=False)

print('=== TOP 20 combinations by PR-AUC ===')
display(
    df_res_sorted.head(20)
    [['Variant', 'Data_Type', 'Technique', 'Ratio', 'Recall', 'Precision', 'F1', 'ROC_AUC', 'PR_AUC', 'Threshold']]
    .style.background_gradient(cmap='YlOrRd', subset=['PR_AUC', 'Recall', 'ROC_AUC'])
    .format({'Recall': '{:.3f}', 'Precision': '{:.3f}', 'F1': '{:.3f}',
             'ROC_AUC': '{:.3f}', 'PR_AUC': '{:.3f}', 'Threshold': '{:.4f}'})
)

### 7.2 Best configuration per variant

In [ ]:
best_per_variant = (
    df_res.sort_values('PR_AUC', ascending=False)
    .groupby(['Variant', 'Data_Type'], as_index=False)
    .first()
)

print('=== Best dataset per variant and data type ===')
display(
    best_per_variant
    [['Variant', 'Data_Type', 'Technique', 'Ratio', 'Recall', 'Precision', 'F1', 'PR_AUC']]
    .sort_values(['Data_Type', 'PR_AUC'], ascending=[True, False])
    .style.background_gradient(cmap='Blues', subset=['PR_AUC'])
    .format({'Recall': '{:.3f}', 'Precision': '{:.3f}', 'F1': '{:.3f}', 'PR_AUC': '{:.3f}'})
)

### 7.3 Visual comparison of variants

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

metrics = ['PR_AUC', 'Recall', 'F1', 'ROC_AUC']
titles  = ['PR-AUC (main metric)', 'Recall', 'F1-Score', 'ROC-AUC']

for ax, metric, title in zip(axes.flat, metrics, titles):
    pivot = (
        df_res.groupby(['Variant', 'Data_Type'])[metric]
        .mean()
        .reset_index()
    )
    sns.barplot(data=pivot, x='Variant', y=metric, hue='Data_Type',
                palette={'augmented': '#3498db', 'original': '#e74c3c'}, ax=ax)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    ax.legend(title='Data type')

plt.suptitle('XGBoost variant comparison — Mean over all datasets',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, data_type in zip(axes, ['augmented', 'original']):
    sub = df_res[df_res['Data_Type'] == data_type]
    pivot = sub.groupby(['Variant', 'Technique'])['PR_AUC'].mean().reset_index()
    pivot_wide = pivot.pivot(index='Technique', columns='Variant', values='PR_AUC')
    sns.heatmap(pivot_wide, annot=True, fmt='.3f', cmap='YlOrRd',
                linewidths=0.5, ax=ax, cbar_kws={'label': 'PR-AUC'})
    ax.set_title(f'PR-AUC — {data_type.capitalize()} data\n(average over ratios)',
                 fontweight='bold')
    ax.set_xlabel('XGBoost variant')
    ax.set_ylabel('Balancing technique')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('PR-AUC Heatmap: Variant × Balancing Technique', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Best Global Model Analysis
### 8.1 Confusion matrix

In [ ]:
best_row = df_res_sorted.iloc[0]
print(f"Best global combination (PR-AUC = {best_row['PR_AUC']:.4f})")
print(f"  Variant   : {best_row['Variant']}")
print(f"  Data type : {best_row['Data_Type']}")
print(f"  Technique : {best_row['Technique']}")
print(f"  Ratio     : {best_row['Ratio']}")
print(f"  Threshold : {best_row['Threshold']}")
print(f"  S3        : {best_row['S3_Path']}")

# Load wrapper from S3
parsed   = urlparse(best_row['S3_Path'])
s3_key   = parsed.path.lstrip('/')
buf      = BytesIO()
s3_client.download_fileobj(BUCKET, s3_key, buf)
buf.seek(0)
best_wrapper = joblib.load(buf)

# Correct holdout depending on data_type
if best_row['Data_Type'] == 'augmented':
    X_te, y_te = X_test_aug[best_wrapper.feature_names], y_test_aug
else:
    feat_ok = [f for f in best_wrapper.feature_names if f in X_test_orig.columns]
    X_te, y_te = X_test_orig[feat_ok], y_test_orig

y_prob_best = best_wrapper.model.predict_proba(X_te.values)[:, 1]
y_pred_best = (y_prob_best >= best_wrapper.threshold).astype(int)
cm = confusion_matrix(y_te, y_pred_best)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Legitimate', 'Vishing'], yticklabels=['Legitimate', 'Vishing'])
axes[0].set_title('Counts')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=axes[1],
            xticklabels=['Legitimate', 'Vishing'], yticklabels=['Legitimate', 'Vishing'])
axes[1].set_title('Normalized')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.suptitle(f"{best_row['Variant']} · {best_row['Technique']} {best_row['Ratio']} · {best_row['Data_Type']}",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nTN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}')
print(f'Recall    : {tp/(tp+fn):.4f}')
print(f'Precision : {tp/(tp+fp):.4f}')
print(f'F1        : {2*tp/(2*tp+fp+fn):.4f}')
print(f'PR-AUC    : {average_precision_score(y_te, y_prob_best):.4f}')

### 8.2 Precision-Recall curve and threshold analysis

In [ ]:
prec_c, rec_c, thr_c = precision_recall_curve(y_te, y_prob_best)
f1_c   = 2 * prec_c[:-1] * rec_c[:-1] / (prec_c[:-1] + rec_c[:-1] + 1e-9)
best_i = np.argmax(f1_c)
pr_auc = average_precision_score(y_te, y_prob_best)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(rec_c, prec_c, color='#3498db', linewidth=2, label=f'PR-AUC = {pr_auc:.4f}')
axes[0].fill_between(rec_c, prec_c, alpha=0.08, color='#3498db')
axes[0].axhline(y=y_te.mean(), color='gray', linestyle='--', alpha=0.7,
                label=f'Baseline ({y_te.mean():.4f})')
axes[0].scatter([rec_c[best_i]], [prec_c[best_i]], s=100, color='red', zorder=5,
                label=f'F1 optimum (thr={thr_c[best_i]:.3f})')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1.05)

axes[1].plot(thr_c, prec_c[:-1], label='Precision', color='#2ecc71', linewidth=2)
axes[1].plot(thr_c, rec_c[:-1],  label='Recall',    color='#e74c3c', linewidth=2)
axes[1].plot(thr_c, f1_c,        label='F1',        color='#f39c12', linewidth=2, linestyle='--')
axes[1].axvline(x=thr_c[best_i], color='black', linestyle=':', alpha=0.8,
                label=f'F1 optimum = {thr_c[best_i]:.3f}')
axes[1].axvline(x=0.5, color='gray', linestyle=':', alpha=0.5, label='Default (0.5)')
axes[1].set_xlabel('Decision threshold')
axes[1].set_ylabel('Metric')
axes[1].set_title('Precision / Recall / F1 vs Threshold', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1.05)

plt.suptitle(f"Threshold Analysis — {best_row['Variant']}", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 8.3 Feature Importance of the best model

In [ ]:
fi = pd.DataFrame({
    'feature'   : best_wrapper.feature_names,
    'importance': best_wrapper.model.feature_importances_,
}).sort_values('importance', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 10))
top25   = fi.head(25)

q80 = top25['importance'].quantile(0.80)
q60 = top25['importance'].quantile(0.60)
colors = ['#e74c3c' if v >= q80 else '#f39c12' if v >= q60 else '#3498db'
          for v in top25['importance']]

ax.barh(range(len(top25)), top25['importance'].values,
        color=colors, edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(top25)))
ax.set_yticklabels(top25['feature'].values)
ax.invert_yaxis()
ax.set_xlabel('Importance (Gain)')
ax.set_title(f'Feature Importance — {best_row["Variant"]}  |  Top 25', fontweight='bold', fontsize=13)

for i, v in enumerate(top25['importance'].values):
    ax.text(v + top25['importance'].max() * 0.005, i, f'{v:.4f}', va='center', fontsize=8)

ax.legend(handles=[
    mpatches.Patch(facecolor='#e74c3c', label='Top 20%'),
    mpatches.Patch(facecolor='#f39c12', label='Top 20-40%'),
    mpatches.Patch(facecolor='#3498db', label='Rest'),
], loc='lower right')

plt.tight_layout()
plt.show()

print('Top 15 features:')
print(f'{"Feature":40s} {"Importance":>12s}')
print('-' * 55)
for _, row in fi.head(15).iterrows():
    print(f"{row['feature']:40s} {row['importance']:12.4f}")

## 9. Executive summary

In [ ]:
print('=' * 70)
print('TRAINING SUMMARY')
print('=' * 70)
print(f'  Total models trained     : {len(df_res)}')
print(f'  XGBoost variants         : {df_res["Variant"].nunique()}')
print(f'  Augmented datasets       : {df_res[df_res["Data_Type"]=="augmented"]["Technique"].count() // df_res["Variant"].nunique()}')
print(f'  Original datasets        : {df_res[df_res["Data_Type"]=="original"]["Technique"].count() // df_res["Variant"].nunique()}')
print(f'  Models in S3             : s3://{BUCKET}/{MODELS_PREFIX}/')
print()

for data_type in ['augmented', 'original']:
    sub  = df_res[df_res['Data_Type'] == data_type]
    best = sub.sort_values('PR_AUC', ascending=False).iloc[0]
    print(f'  Best {data_type:10s} — {best["Variant"]:20s} | {best["Technique"]:25s} | ratio={best["Ratio"]:4s}')
    print(f'    PR-AUC={best["PR_AUC"]:.4f}  Recall={best["Recall"]:.4f}  F1={best["F1"]:.4f}')
    print()

print('=' * 70)
print(f'  Best global → {best_row["Variant"]} / {best_row["Data_Type"]} / {best_row["Technique"]} / {best_row["Ratio"]}')
print(f'  PR-AUC={best_row["PR_AUC"]:.4f}  Recall={best_row["Recall"]:.4f}  F1={best_row["F1"]:.4f}')
print('=' * 70)

# Save results table to S3
csv_buf = BytesIO()
df_res.sort_values('PR_AUC', ascending=False).to_csv(csv_buf, index=False)
csv_buf.seek(0)
results_key = f'{MODELS_PREFIX}/resultados_xgb_experimento.csv'
s3_client.upload_fileobj(csv_buf, BUCKET, results_key)
print(f'\nResults table saved to: s3://{BUCKET}/{results_key}')